# Chemical Space Explorer — Framework 1: Structural Fingerprints

## Mission
Identify structurally distinct scaffold families across a compound library to guide
combinatorial building block selection. The goal is to select ~30 building blocks from
millions of enumerated structures such that their combinations maximally cover structural
chemical space. K-Medoids medoids are the synthesis candidates — one representative
scaffold archetype per cluster.

## Representation
Morgan fingerprints (radius 2, 2048 bits) capture circular substructure environments.
Tanimoto similarity is used as the chemical similarity metric wherever raw fingerprints
are available (deduplication, inter-cluster diversity, Murcko validation).

At 5M+ compound scale, a full pairwise Tanimoto distance matrix (~25 trillion entries)
is not feasible. PCA compresses 2048-bit fingerprints to ~50 principal components
before clustering. After PCA, cosine/euclidean distance is used — this is a standard
approximation of Tanimoto distance in compressed fingerprint space. This is disclosed
explicitly in the methods.

## Pipeline
```
SMILES → Morgan FP (2048-bit)
  → Tanimoto deduplication (exact + near-duplicate removal)
  → PCA elbow → PCA-50 compressed space
  → K-Medoids (cosine, k from elbow) ─┐ cluster in
  → HDBSCAN (euclidean)               ┘ PCA-50 space
  → UMAP (cosine, 1 seed) → 2D canvas (visualization only)
  → Paint cluster labels onto UMAP canvas
```

## QC Scores
- Silhouette (cosine) → K-Medoids quality
- DBCV (relative_validity_) → HDBSCAN quality
- PCA variance explained → compression quality
- Tanimoto intra/inter-cluster diversity → cluster chemical meaningfulness

## Outputs
- `cluster_assignments_fp.csv` — per-compound cluster labels + UMAP XY
- `synthesis_candidates_fp.csv` — ranked medoids with coverage score
- 6 figures

---
**Note — future experiments (flag for follow-up):**
- HDBSCAN parameter sensitivity grid (min_cluster_size × min_samples): omitted here
  because 12-run grid on 5M compounds ≈ 8 hrs on A100. Run on subsampled dataset first.
- UMAP 5-seed stability check: omitted because 5 UMAP runs × 5M compounds ≈ 2.5 hrs.
  Single canonical seed used here. Re-enable on subsampled data to validate.
- Bootstrap cluster stability (80% subsampling × 20 runs): deferred to downstream
  validation after initial results reviewed.
- Full Tanimoto clustering (without PCA compression): feasible on subsets < 50K compounds.

In [ ]:
!pip install -q umap-learn hdbscan scikit-learn-extra rdkit-pypi
print('Dependencies installed.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — only edit this cell
# ═══════════════════════════════════════════════════════════════════════════════

LIBRARY_CSV    = 'library.csv'       # compound library
LITERATURE_CSV = 'literature.csv'    # known actives reference (None to skip)

SMILES_COL     = 'smiles'
NAME_COL       = 'name'             # None if absent
ACTIVITY_COL   = 'ic50_nm'         # None to skip activity overlay
ACTIVITY_LABEL = 'IC50 (nM)'
ACTIVITY_LOG   = True               # log10-transform activity

# Morgan fingerprint parameters
FP_RADIUS      = 2
FP_NBITS       = 2048

# PCA compression: number of components to try in elbow plot
PCA_MAX_COMPONENTS = 100
# After elbow inspection, set this to the chosen number of components:
PCA_N_COMPONENTS   = 50  # override after running the elbow cell

# K-Medoids: set after running the elbow plot cell
N_KMEDOIDS     = 8       # override after elbow

# HDBSCAN
HDBSCAN_MIN_SIZE  = 50
HDBSCAN_MIN_SAMP  = 10

# UMAP (visualization only — not used for clustering)
UMAP_N_NEIGHBORS  = 30
UMAP_MIN_DIST     = 0.1
RANDOM_STATE      = 42
# Note: single seed used here due to scale (5M compounds). For stability
# validation, run check_umap_stability() on a subsampled dataset (~50K).

# Tanimoto deduplication threshold
DEDUP_TANIMOTO = 0.95    # compounds with sim > threshold are near-duplicates

# Inter-cluster diversity: sample size per cluster (full pairwise not feasible at 5M)
DIVERSITY_SAMPLE = 1000

# Maximum compounds to process (None = all). Set to e.g. 500_000 for fast test runs.
MAX_COMPOUNDS  = None

OUTPUT_DIR     = 'results_fp'
# ═══════════════════════════════════════════════════════════════════════════════
print('Configuration loaded.')

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

import umap
import hdbscan as hdbscan_lib
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn_extra.cluster import KMedoids

from rdkit import Chem
from rdkit.Chem import AllChem, Draw, DataStructs, Scaffolds
from rdkit.Chem.Scaffolds import MurckoScaffold
from IPython.display import display

warnings.filterwarnings('ignore')
Path(OUTPUT_DIR).mkdir(exist_ok=True)
print('Imports OK.')

## 1. Load Data

In [ ]:
def load_dataset(path, label):
    df = pd.read_csv(path)
    df = df.rename(columns={SMILES_COL: 'smiles'})
    if NAME_COL and NAME_COL in df.columns:
        df = df.rename(columns={NAME_COL: 'name'})
    else:
        df['name'] = [f'{label}_{i}' for i in range(len(df))]
    if ACTIVITY_COL and ACTIVITY_COL in df.columns:
        df = df.rename(columns={ACTIVITY_COL: 'activity'})
    df['source'] = label
    return df

lib_df = load_dataset(LIBRARY_CSV, 'library')
if LITERATURE_CSV:
    lit_df = load_dataset(LITERATURE_CSV, 'literature')
    df = pd.concat([lib_df, lit_df], ignore_index=True)
else:
    df = lib_df.copy()

if MAX_COMPOUNDS and len(df) > MAX_COMPOUNDS:
    print(f'Subsampling to {MAX_COMPOUNDS:,} compounds (MAX_COMPOUNDS set).')
    df = df.sample(MAX_COMPOUNDS, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Loaded: {len(df):,} compounds')
print(f'Sources: {df["source"].value_counts().to_dict()}')

## 2. Deduplication

Remove exact SMILES duplicates first, then near-duplicates (Tanimoto > threshold).
Near-duplicate removal uses a greedy approach: sort by source (keep literature compounds
preferentially), then remove any compound with Tanimoto > threshold against an already-
kept compound. This prevents artificial inflation of cluster quality.

In [ ]:
n_before = len(df)

# Step 1: exact SMILES duplicates
df = df.drop_duplicates(subset='smiles').reset_index(drop=True)
n_exact = n_before - len(df)
print(f'Exact duplicates removed: {n_exact:,}')

# Step 2: compute Morgan fingerprints
print('Computing Morgan fingerprints...')
fps = []
valid_mask = []
for smi in df['smiles']:
    mol = Chem.MolFromSmiles(str(smi))
    if mol:
        fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, FP_RADIUS, nBits=FP_NBITS))
        valid_mask.append(True)
    else:
        fps.append(None)
        valid_mask.append(False)

n_parse_fail = valid_mask.count(False)
if n_parse_fail:
    print(f'SMILES parse failures: {n_parse_fail} — these rows will be excluded.')
    df['_parse_ok'] = valid_mask
    df.loc[~df['_parse_ok'], 'smiles'].to_csv(f'{OUTPUT_DIR}/parse_failures.csv', index=False)
    df = df[df['_parse_ok']].reset_index(drop=True)
    fps = [fp for fp in fps if fp is not None]

df['_fp'] = fps

# Step 3: near-duplicate removal
# Greedy O(n²) is not feasible at 5M. Use batched approach: compare each compound
# against a random sample of already-kept compounds. Exact near-dedup would require
# LSH (locality-sensitive hashing) at this scale — flagged as future improvement.
print(f'Near-duplicate removal (Tanimoto > {DEDUP_TANIMOTO})...')
# Sort: literature first (preferentially retained)
df = df.sort_values('source', ascending=False).reset_index(drop=True)
fps_list = df['_fp'].tolist()

keep = [True] * len(df)
kept_fps = []
BATCH_CHECK = 200  # compare against last N kept fps (memory-efficient approximation)
for i, fp in enumerate(fps_list):
    if not keep[i]:
        continue
    if kept_fps:
        sims = DataStructs.BulkTanimotoSimilarity(fp, kept_fps[-BATCH_CHECK:])
        if max(sims) > DEDUP_TANIMOTO:
            keep[i] = False
            continue
    kept_fps.append(fp)

df = df[keep].reset_index(drop=True)
fps = df['_fp'].tolist()
n_neardup = sum(~np.array(keep))
print(f'Near-duplicates removed: {n_neardup:,}')
print(f'Final dataset: {len(df):,} compounds')

if 'activity' in df.columns:
    df['activity_plot'] = np.log10(df['activity'].clip(lower=1e-3)) if ACTIVITY_LOG else df['activity']

## 3. Dataset Characterization

Before clustering: understand what you are working with.
- Source breakdown and activity distribution
- Tanimoto similarity distribution (how diverse is the library overall?)
- A flat Tanimoto distribution (all pairs ~0.2) = structurally diverse library
- A peaked distribution near 1.0 = redundant library dominated by one scaffold family

In [ ]:
print('=== Dataset Summary ===')
print(f'Total compounds: {len(df):,}')
print(f'Sources: {df["source"].value_counts().to_dict()}')
if 'activity' in df.columns:
    act = df['activity'].dropna()
    print(f'Activity ({ACTIVITY_LABEL}): n={len(act):,}  median={act.median():.2f}  range=[{act.min():.2f}, {act.max():.2f}]')

# Tanimoto distribution — sample 2000 random pairs (full pairwise not feasible at scale)
print('\nSampling 2000 random Tanimoto pairs for diversity assessment...')
sample_idx = np.random.choice(len(fps), size=min(2000, len(fps)), replace=False)
sample_fps = [fps[i] for i in sample_idx]
tan_sims = []
for i in range(len(sample_fps)):
    for j in range(i+1, len(sample_fps)):
        tan_sims.append(DataStructs.TanimotoSimilarity(sample_fps[i], sample_fps[j]))
tan_sims = np.array(tan_sims)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(tan_sims, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(np.median(tan_sims), color='red', linestyle='--', label=f'Median={np.median(tan_sims):.3f}')
axes[0].set_xlabel('Pairwise Tanimoto Similarity'); axes[0].set_ylabel('Count')
axes[0].set_title('Library Structural Diversity\n(sampled 2K pairs)')
axes[0].legend()
# Interpretation guide
axes[0].text(0.02, 0.95,
    'Flat/left-skewed = diverse library\nRight-peaked = redundant library',
    transform=axes[0].transAxes, fontsize=8, va='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

if 'activity' in df.columns and df['activity'].notna().sum() > 0:
    df['activity_plot'].dropna().hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
    axes[1].set_xlabel(f'log10({ACTIVITY_LABEL})' if ACTIVITY_LOG else ACTIVITY_LABEL)
    axes[1].set_ylabel('Count')
    axes[1].set_title('Activity Distribution')
else:
    axes[1].text(0.5, 0.5, 'No activity data', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dataset_characterization.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nMedian pairwise Tanimoto: {np.median(tan_sims):.3f}')
print(f'Fraction pairs with Tanimoto > 0.4: {(tan_sims > 0.4).mean():.3f}')

## 4. PCA Compression: 2048-bit → N components

Morgan fingerprints are 2048-dimensional binary vectors. Direct Tanimoto clustering
at 5M+ scale requires a precomputed distance matrix (~100 TB) — not feasible.

PCA compresses the fingerprint space to ~50 continuous components that capture the
majority of structural variance. After PCA, cosine/euclidean distance is used for
clustering — a standard approximation to Tanimoto in compressed fingerprint space.

**Elbow rule**: choose the number of components where the cumulative explained variance
curve flattens (the 'elbow'). Typical values: 30–80 components for drug-like libraries.

In [ ]:
# Convert fingerprints to numpy array
print('Converting fingerprints to array...')
fp_array = np.array([list(fp) for fp in fps], dtype=np.float32)
print(f'Fingerprint matrix: {fp_array.shape}')

# PCA elbow plot — run on a sample if dataset is very large
n_for_pca = min(len(fp_array), 100_000)
if n_for_pca < len(fp_array):
    print(f'PCA elbow computed on {n_for_pca:,} sample (full dataset too large for elbow sweep).')
    pca_sample = fp_array[np.random.choice(len(fp_array), n_for_pca, replace=False)]
else:
    pca_sample = fp_array

n_comp = min(PCA_MAX_COMPONENTS, pca_sample.shape[1], pca_sample.shape[0] - 1)
pca_full = PCA(n_components=n_comp, random_state=RANDOM_STATE)
pca_full.fit(pca_sample)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, n_comp+1), cumvar, 'o-', markersize=3, color='steelblue')
ax.axhline(0.80, color='orange', linestyle='--', label='80% variance')
ax.axhline(0.90, color='red', linestyle='--', label='90% variance')
ax.axvline(PCA_N_COMPONENTS, color='green', linestyle='--',
           label=f'Current PCA_N_COMPONENTS={PCA_N_COMPONENTS}')
ax.set_xlabel('Number of PCA Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA Elbow — Choose PCA_N_COMPONENTS at the elbow')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pca_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

var_at_n = cumvar[PCA_N_COMPONENTS - 1]
print(f'\nVariance explained by {PCA_N_COMPONENTS} components: {var_at_n:.1%}')
print('If < 80%, increase PCA_N_COMPONENTS in the config cell and rerun.')

In [ ]:
# Apply PCA compression to full dataset
pca = PCA(n_components=PCA_N_COMPONENTS, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(fp_array)
print(f'PCA-compressed matrix: {X_pca.shape}')
print(f'Variance explained: {pca.explained_variance_ratio_.sum():.1%}')

## 5. Clustering in PCA-Compressed Fingerprint Space

All clustering is performed on the PCA-compressed descriptor matrix (not UMAP coordinates).
UMAP runs later for visualization only.

Cosine distance is used for K-Medoids: it measures the angle between fingerprint vectors,
invariant to overall fingerprint density differences across compounds of different sizes —
analogous to Tanimoto in the compressed space.

In [ ]:
# ── K-Medoids elbow: silhouette vs k ─────────────────────────────────────────
# Run on a sample if dataset is large (K-Medoids scales O(n²) with n compounds)
n_elbow = min(len(X_pca), 10_000)
X_elbow = X_pca[np.random.choice(len(X_pca), n_elbow, replace=False)] if n_elbow < len(X_pca) else X_pca
print(f'Elbow plot computed on {n_elbow:,} sample.')

k_range = range(2, 21)
sil_scores = []
for k in k_range:
    km_test = KMedoids(n_clusters=k, metric='cosine', method='alternate', random_state=RANDOM_STATE)
    lbl = km_test.fit_predict(X_elbow)
    sil_scores.append(silhouette_score(X_elbow, lbl, metric='cosine'))
    print(f'  k={k:2d}  silhouette={sil_scores[-1]:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), sil_scores, 'o-', color='steelblue', markersize=6)
ax.axvline(N_KMEDOIDS, color='red', linestyle='--', label=f'Current N_KMEDOIDS={N_KMEDOIDS}')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Silhouette Score (cosine)')
ax.set_title('K-Medoids Elbow — Choose k at the peak or first plateau')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/kmedoids_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nOptimal k (max silhouette): {list(k_range)[np.argmax(sil_scores)]}')
print('Update N_KMEDOIDS in the config cell if needed, then rerun next cell.')

In [ ]:
# ── K-Medoids on full PCA-compressed dataset ──────────────────────────────────
print(f'Running K-Medoids (k={N_KMEDOIDS}, metric=cosine)...')
km = KMedoids(
    n_clusters   = N_KMEDOIDS,
    metric       = 'cosine',
    method       = 'alternate',
    random_state = RANDOM_STATE,
)
km_labels  = km.fit_predict(X_pca)
medoid_idx = list(km.medoid_indices_)

sil_km = silhouette_score(X_pca, km_labels, metric='cosine')
print(f'K-Medoids done.')
print(f'  Silhouette (cosine): {sil_km:.4f}')
print(f'  Interpretation: {"strong" if sil_km > 0.5 else "reasonable" if sil_km > 0.25 else "weak"}')
df['km_cluster'] = km_labels

In [ ]:
# ── HDBSCAN on PCA-compressed dataset ────────────────────────────────────────
# DBCV (relative_validity_) is the correct quality metric for HDBSCAN.
# Silhouette assumes convex clusters; DBCV measures density within vs between
# clusters and handles arbitrary cluster shapes. Range: -1 (worst) to +1 (best).
print('Running HDBSCAN...')
clusterer = hdbscan_lib.HDBSCAN(
    min_cluster_size       = HDBSCAN_MIN_SIZE,
    min_samples            = HDBSCAN_MIN_SAMP,
    metric                 = 'euclidean',
    cluster_selection_method = 'eom',
    gen_min_span_tree      = True,  # required for DBCV
    core_dist_n_jobs       = -1,    # use all CPU cores
)
hdb_labels = clusterer.fit_predict(X_pca)

n_hdb_clusters = len(set(hdb_labels) - {-1})
n_noise        = (hdb_labels == -1).sum()
dbcv           = float(clusterer.relative_validity_)

print(f'HDBSCAN done.')
print(f'  Clusters: {n_hdb_clusters}  |  Noise: {n_noise} ({100*n_noise/len(df):.1f}%)')
print(f'  DBCV: {dbcv:.4f}')
print(f'  Interpretation: {"well-separated" if dbcv > 0.5 else "moderate" if dbcv > 0 else "poor"}')

# Note: HDBSCAN sensitivity grid (min_cluster_size × min_samples) omitted at this scale.
# Run on a 50K subsample to tune parameters before full-scale deployment.

df['hdb_cluster'] = hdb_labels

## 6. UMAP — Visualization Only

UMAP compresses the PCA-50 matrix to 2D for plotting. It is applied **after** clustering
and its output is **never used as input to any algorithm**.

Single canonical seed (RANDOM_STATE=42) is used at this scale. For stability validation,
run on a subsampled dataset (~50K compounds) with 5 seeds and compute pairwise ARI
to confirm the layout is reproducible before interpreting cluster boundaries.

In [ ]:
print('Running UMAP (visualization only)...')
reducer = umap.UMAP(
    n_neighbors  = UMAP_N_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    n_components = 2,
    metric       = 'cosine',
    random_state = RANDOM_STATE,
    low_memory   = True,  # required for large datasets
)
embedding = reducer.fit_transform(X_pca)
df['umap_x'] = embedding[:, 0]
df['umap_y'] = embedding[:, 1]
print('UMAP done.')

## 7. Visualization

All plots share the same UMAP 2D canvas. Cluster labels come from K-Medoids and HDBSCAN
(PCA-compressed fingerprint space). UMAP coordinates are used only for plotting.

In [ ]:
# ── Plot 1: Library vs Literature ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for src in df['source'].unique():
    mask = df['source'] == src
    ax.scatter(df.loc[mask,'umap_x'], df.loc[mask,'umap_y'],
               s=120 if src=='literature' else 6,
               marker='*' if src=='literature' else 'o',
               alpha=0.7, label=f'{src} (n={mask.sum():,})',
               zorder=3 if src=='literature' else 2,
               edgecolors='white' if src=='literature' else 'none', linewidths=0.5)
ax.set_title('Chemical Space — Library vs Literature Reference', fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot1_source.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: K-Medoids Clusters ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
cmap = plt.cm.tab10
for lab in sorted(df['km_cluster'].unique()):
    mask = df['km_cluster'] == lab
    ax.scatter(df.loc[mask,'umap_x'], df.loc[mask,'umap_y'],
               c=[cmap(int(lab)%10)], s=6, alpha=0.4,
               label=f'K{lab} (n={mask.sum():,})', rasterized=True)
ax.scatter(df.iloc[medoid_idx]['umap_x'], df.iloc[medoid_idx]['umap_y'],
           s=200, marker='*', c='black', zorder=10,
           edgecolors='white', linewidths=0.8, label='Medoids (synthesis candidates)')
for idx in medoid_idx:
    ax.annotate(f'K{km_labels[idx]}',
                (df.iloc[idx]['umap_x'], df.iloc[idx]['umap_y']),
                fontsize=7, ha='center', xytext=(0,6), textcoords='offset points')
ax.set_title(f'Track A — K-Medoids Archetypes (k={N_KMEDOIDS})\nSilhouette (cosine): {sil_km:.3f}', fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot2_kmedoids.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: HDBSCAN Natural Clusters ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
noise_mask = df['hdb_cluster'] == -1
if noise_mask.sum():
    ax.scatter(df.loc[noise_mask,'umap_x'], df.loc[noise_mask,'umap_y'],
               c='lightgrey', s=4, alpha=0.2, label=f'Noise (n={noise_mask.sum():,})', rasterized=True)
cmap_hdb = plt.cm.tab20
for lab in sorted(set(df['hdb_cluster'].unique()) - {-1}):
    mask = df['hdb_cluster'] == lab
    ax.scatter(df.loc[mask,'umap_x'], df.loc[mask,'umap_y'],
               c=[cmap_hdb(int(lab)%20)], s=8, alpha=0.6,
               label=f'C{lab} (n={mask.sum():,})', rasterized=True)
ax.set_title(f'Track B — HDBSCAN Natural Clusters ({n_hdb_clusters})\nDBCV: {dbcv:.3f}', fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot3_hdbscan.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 4: Activity Overlay ───────────────────────────────────────────────────
# Literature IC50/DC50 data serves as external validation:
# if active compounds cluster together, the fingerprint space captures activity-relevant
# structural variation.
if 'activity_plot' in df.columns and df['activity_plot'].notna().sum() > 0:
    fig, ax = plt.subplots(figsize=(9, 6))
    has_act = df['activity_plot'].notna()
    ax.scatter(df.loc[~has_act,'umap_x'], df.loc[~has_act,'umap_y'],
               c='lightgrey', s=4, alpha=0.15, rasterized=True, label='No activity data')
    vals = df.loc[has_act,'activity_plot']
    norm = mcolors.Normalize(vmin=np.percentile(vals,5), vmax=np.percentile(vals,95))
    sc = ax.scatter(df.loc[has_act,'umap_x'], df.loc[has_act,'umap_y'],
                    c=vals, cmap='RdYlGn_r', norm=norm, s=10, alpha=0.8, rasterized=True)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(f'log10({ACTIVITY_LABEL})' if ACTIVITY_LOG else ACTIVITY_LABEL)
    ax.set_title(f'External Validation — {ACTIVITY_LABEL}\n(green=potent, red=inactive)', fontweight='bold')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plot4_activity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No activity data — skipping Plot 4.')

## 8. Cluster Diversity Analysis

**Intra-cluster Tanimoto**: mean pairwise similarity within each cluster.
- High (> 0.7): tight scaffold family — one representative sufficient
- Low (< 0.4): diverse cluster — worth sampling multiple compounds

**Inter-cluster Tanimoto**: mean pairwise similarity between clusters.
- Low inter-cluster similarity = clusters are genuinely distinct regions of chemical space ✓
- High inter-cluster similarity = clusters are too similar and may be over-partitioned;
  consider reducing k or relaxing HDBSCAN parameters

Computed on a sample of DIVERSITY_SAMPLE compounds per cluster (full pairwise not
feasible at 5M+ scale).

In [ ]:
# Sample fingerprints per cluster
cluster_fps = {}
for lab in sorted(df['km_cluster'].unique()):
    idx = df[df['km_cluster'] == lab].index.tolist()
    sampled = np.random.choice(idx, size=min(DIVERSITY_SAMPLE, len(idx)), replace=False)
    cluster_fps[lab] = [fps[i] for i in sampled]

# Intra-cluster similarity
intra_sim = {}
for lab, cfps in cluster_fps.items():
    sims = [DataStructs.TanimotoSimilarity(cfps[i], cfps[j])
            for i in range(len(cfps)) for j in range(i+1, len(cfps))]
    intra_sim[lab] = np.mean(sims) if sims else np.nan

# Inter-cluster similarity matrix
labs = sorted(cluster_fps.keys())
n_labs = len(labs)
inter_matrix = np.zeros((n_labs, n_labs))
for i, li in enumerate(labs):
    for j, lj in enumerate(labs):
        if i == j:
            inter_matrix[i,j] = intra_sim[li]
        else:
            fps_i = cluster_fps[li][:200]  # further limit for inter-cluster
            fps_j = cluster_fps[lj][:200]
            sims = [DataStructs.TanimotoSimilarity(fi, fj) for fi in fps_i for fj in fps_j]
            inter_matrix[i,j] = np.mean(sims)

# ── Plot 5: Diversity Heatmap ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(inter_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Mean Tanimoto Similarity')
tick_labels = [f'K{lab}' for lab in labs]
ax.set_xticks(range(n_labs)); ax.set_yticks(range(n_labs))
ax.set_xticklabels(tick_labels, rotation=45, ha='right')
ax.set_yticklabels(tick_labels)
for i in range(n_labs):
    for j in range(n_labs):
        ax.text(j, i, f'{inter_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title(
    'Cluster Similarity Heatmap (Tanimoto)\n'
    'Diagonal = intra-cluster | Off-diagonal = inter-cluster\n'
    'High off-diagonal = clusters too similar (consider reducing k)',
    fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot5_diversity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Intra-cluster mean Tanimoto:')
for lab, sim in intra_sim.items():
    print(f'  K{lab}: {sim:.3f}  → {"tight scaffold" if sim > 0.7 else "diverse" if sim < 0.4 else "moderate"}')

## 9. Murcko Scaffold Enrichment

A Bemis-Murcko scaffold is the core ring system + linkers of a molecule, with all
side chains stripped. Two molecules sharing a Murcko scaffold have the same core
skeleton regardless of substituents.

If a cluster has high Murcko scaffold concentration (many compounds share one scaffold),
it is a tight scaffold family — one representative is sufficient for synthesis.
If a cluster has many unique Murcko scaffolds, it is genuinely structurally diverse.

In [ ]:
print('Computing Murcko scaffolds...')
scaffolds = []
for smi in df['smiles']:
    mol = Chem.MolFromSmiles(str(smi))
    if mol:
        try:
            scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        except:
            scaf = ''
    else:
        scaf = ''
    scaffolds.append(scaf)
df['murcko_scaffold'] = scaffolds

print('\nMurcko scaffold analysis per cluster:')
scaffold_rows = []
for lab in sorted(df['km_cluster'].unique()):
    sub = df[df['km_cluster'] == lab]
    n_total = len(sub)
    n_unique = sub['murcko_scaffold'].nunique()
    top_scaf = sub['murcko_scaffold'].value_counts().head(3)
    top_pct = top_scaf.iloc[0] / n_total if len(top_scaf) else 0
    scaffold_rows.append({
        'cluster':       f'K{lab}',
        'n_compounds':   n_total,
        'n_unique_scaffolds': n_unique,
        'scaffold_diversity': round(n_unique / n_total, 3),
        'top_scaffold_pct':   round(top_pct, 3),
        'top_scaffold':       top_scaf.index[0] if len(top_scaf) else '',
    })
    print(f'  K{lab}: {n_unique} unique scaffolds / {n_total} compounds'
          f'  (diversity={n_unique/n_total:.2f}, top scaffold covers {top_pct:.1%})')

scaffold_df = pd.DataFrame(scaffold_rows)
scaffold_df.to_csv(f'{OUTPUT_DIR}/murcko_scaffold_enrichment.csv', index=False)

In [ ]:
# ── Plot 6: Medoid Structure Grid ─────────────────────────────────────────────
# These are the synthesis candidates — one per K-Medoids cluster.
medoid_mols, medoid_labels = [], []
for idx in medoid_idx:
    row = df.iloc[idx]
    mol = Chem.MolFromSmiles(str(row['smiles']))
    if mol:
        AllChem.Compute2DCoords(mol)
        medoid_mols.append(mol)
        act_str = ''
        if 'activity' in df.columns and pd.notna(row.get('activity')):
            act_str = f'\n{ACTIVITY_LABEL}={row["activity"]:.1f}'
        intra = intra_sim.get(km_labels[idx], np.nan)
        medoid_labels.append(f'K{km_labels[idx]} | {row["source"]}\n{row["name"]}{act_str}\nIntra-Tan={intra:.2f}')

img = Draw.MolsToGridImage(
    medoid_mols, molsPerRow=4, subImgSize=(300, 260),
    legends=medoid_labels, returnPNG=False)
img.save(f'{OUTPUT_DIR}/plot6_medoid_structures.png')
display(img)
print('Medoid structure grid saved — these are your synthesis archetypes.')

## 10. Outlier Analysis

HDBSCAN noise compounds (label=-1) do not belong to any dense cluster. These are either:
- Genuinely unusual scaffolds worth investigating as novel starting points
- Low-quality / anomalous entries worth flagging for manual review

Note: bootstrap-based outlier confidence (noise in >50% of subsample runs) is
deferred to downstream validation. See future experiments note in the abstract.

In [ ]:
noise_df = df[df['hdb_cluster'] == -1][['name','smiles','source'] +
            (['activity'] if 'activity' in df.columns else [])].copy()
noise_df['murcko_scaffold'] = df.loc[noise_df.index, 'murcko_scaffold']
print(f'HDBSCAN noise compounds: {len(noise_df):,} ({100*len(noise_df)/len(df):.1f}%)')
print('These compounds do not belong to any dense structural cluster.')
print('Options:')
print('  1. Flag for manual structural review')
print('  2. Reduce HDBSCAN_MIN_SIZE to capture more compounds in clusters')
print('  3. Treat as novel scaffold territory worth separate investigation')
noise_df.to_csv(f'{OUTPUT_DIR}/hdbscan_noise_compounds.csv', index=False)
print(f'\nNoise compound list saved: {OUTPUT_DIR}/hdbscan_noise_compounds.csv')
if len(noise_df) > 0:
    display(noise_df.head(10))

## 11. Synthesis Priority Output

Clusters ranked for building block selection:
1. **Literature overlap** → known active scaffold region (highest confidence)
2. **Library-only, high inter-cluster distance** → novel diverse scaffold
3. **High intra-cluster similarity** → tight family, one representative sufficient

Coverage score: fraction of total fingerprint space (measured in PCA space) that the
selected medoids cover — higher = better combinatorial space coverage.

In [ ]:
rows = []
medoid_pca = X_pca[medoid_idx]  # PCA vectors of medoids

for rank, idx in enumerate(medoid_idx):
    lab  = int(km_labels[idx])
    mask = df['km_cluster'] == lab
    row  = df.iloc[idx]
    n_lib = (df.loc[mask,'source'] == 'library').sum()
    n_lit = (df.loc[mask,'source'] == 'literature').sum() if LITERATURE_CSV else 0
    coverage = 'overlap' if n_lit > 0 and n_lib > 0 else 'literature_only' if n_lit > 0 else 'library_only'

    act_med = float(df.loc[mask,'activity'].median()) if 'activity' in df.columns else None

    # Mean inter-cluster distance from this medoid to all other medoids (cosine)
    from sklearn.metrics.pairwise import cosine_distances
    other_idx = [i for i, m in enumerate(medoid_idx) if m != idx]
    if other_idx:
        dists = cosine_distances(X_pca[[idx]], X_pca[other_idx])[0]
        mean_inter_dist = float(np.mean(dists))
    else:
        mean_inter_dist = np.nan

    rows.append({
        'cluster':           lab,
        'medoid_name':       row['name'],
        'medoid_smiles':     row['smiles'],
        'medoid_source':     row['source'],
        'n_total':           int(mask.sum()),
        'n_library':         int(n_lib),
        'n_literature':      int(n_lit),
        'coverage':          coverage,
        'intra_tanimoto':    round(intra_sim.get(lab, np.nan), 3),
        'mean_inter_dist':   round(mean_inter_dist, 3),
        'murcko_diversity':  scaffold_df.loc[scaffold_df['cluster']==f'K{lab}','scaffold_diversity'].values[0] if f'K{lab}' in scaffold_df['cluster'].values else None,
        'median_activity':   round(act_med, 2) if act_med and not np.isnan(act_med) else None,
    })

synth_df = pd.DataFrame(rows)
coverage_order = {'overlap': 0, 'library_only': 1, 'literature_only': 2}
synth_df['_rank'] = synth_df['coverage'].map(coverage_order)
# Secondary sort: high inter-cluster distance (most distinct medoids first)
synth_df = synth_df.sort_values(['_rank','mean_inter_dist'], ascending=[True, False]).drop(columns='_rank')
synth_df.to_csv(f'{OUTPUT_DIR}/synthesis_candidates_fp.csv', index=False)

print('Synthesis priority ranking:')
display(synth_df[['cluster','medoid_name','coverage','n_total','intra_tanimoto','mean_inter_dist','median_activity']].to_string(index=False))

In [ ]:
# Save per-compound assignments
out_cols = ['name','smiles','source','km_cluster','hdb_cluster','umap_x','umap_y','murcko_scaffold']
if 'activity' in df.columns:
    out_cols.append('activity')
df[out_cols].to_csv(f'{OUTPUT_DIR}/cluster_assignments_fp.csv', index=False)
print(f'Cluster assignments saved: {OUTPUT_DIR}/cluster_assignments_fp.csv')

## 12. QC Summary

In [ ]:
print('=' * 60)
print('QC SUMMARY — Fingerprint Framework')
print('=' * 60)
print(f'Representation        : Morgan FP (r={FP_RADIUS}, {FP_NBITS}-bit)')
print(f'PCA compression       : {FP_NBITS} → {PCA_N_COMPONENTS} components ({pca.explained_variance_ratio_.sum():.1%} variance)')
print(f'Compounds analyzed    : {len(df):,}')
print(f'Deduplication removed : {n_exact:,} exact + {n_neardup:,} near-duplicates')
print()
print(f'K-Medoids (k={N_KMEDOIDS}, cosine)')
print(f'  Silhouette           : {sil_km:.4f}  ({"strong" if sil_km>0.5 else "reasonable" if sil_km>0.25 else "weak"})')
print()
print(f'HDBSCAN')
print(f'  Clusters             : {n_hdb_clusters}')
print(f'  Noise                : {n_noise:,} ({100*n_noise/len(df):.1f}%)')
print(f'  DBCV                 : {dbcv:.4f}  ({"well-separated" if dbcv>0.5 else "moderate" if dbcv>0 else "poor"})')
print()
print(f'UMAP                  : cosine, n_neighbors={UMAP_N_NEIGHBORS}, seed={RANDOM_STATE} (visualization only)')
print(f'Outputs               : {OUTPUT_DIR}/')
print('=' * 60)